In [5]:
minority_indices = [('accountant', 'Female'),
 ('architect', 'Female'),
 ('attorney', 'Female'),
 ('chiropractor', 'Female'),
 ('comedian', 'Female'),
 ('composer', 'Female'),
 ('dentist', 'Female'),
 ('dietitian', 'Male'),
 ('dj', 'Female'),
 ('filmmaker', 'Female'),
 ('interior_designer', 'Male'),
 ('journalist', 'Male'),
 ('model', 'Male'),
 ('nurse', 'Male'),
 ('painter', 'Female'),
 ('paralegal', 'Male'),
 ('pastor', 'Female'),
 ('personal_trainer', 'Female'),
 ('photographer', 'Female'),
 ('physician', 'Female'),
 ('poet', 'Female'),
 ('professor', 'Female'),
 ('psychologist', 'Male'),
 ('rapper', 'Female'),
 ('software_engineer', 'Female'),
 ('surgeon', 'Female'),
 ('teacher', 'Male'),
 ('yoga_teacher', 'Male')]

In [ ]:
import pandas as pd
accuracy_summary = pd.read_csv('./predictions/accuracy_summary_llama2_base.csv')


# Column: ['True Profession', 'True Gender', 'Accuracy', 'Correct Count', 'Total Count']
print(accuracy_summary)


      True Profession True Gender  Accuracy  Correct Count  Total Count
0   interior_designer      Female  0.895349             77           86
1           paralegal      Female  0.900000             81           90
2             dentist        Male  0.897059             61           68
3          accountant        Male  0.901639             55           61
4    personal_trainer      Female  0.885714             31           35
5            attorney        Male  0.900000             54           60
6        chiropractor        Male  0.901235             73           81
7                  dj        Male  0.905882             77           85
8          accountant      Female  0.871795             34           39
9               model      Female  0.896552             78           87
10  software_engineer        Male  0.905882             77           85
11            surgeon        Male  0.905263             86           95
12            teacher        Male  0.888889             32      

In [6]:
import pandas as pd
import matplotlib.pyplot as plt

# Load the two CSV files into DataFrames
accuracy_summary_unbiased_classifier_male = pd.read_csv('./predictions/accuracy_summary_unbiased_male.csv')
accuracy_summary_unbiased_classifier_female = pd.read_csv('./predictions/accuracy_summary_unbiased_classifier_female.csv')
accuracy_summary = pd.read_csv('./predictions/accuracy_summary_llama2_base.csv')

# Print the lengths of the DataFrames
print(f"Length of DataFrame df1: {len(accuracy_summary_unbiased_classifier_male)}")
print(f"Length of DataFrame df2: {len(accuracy_summary)}")
#print(accuracy_summary.head())


# Merge DataFrames on 'True Profession' and 'True Gender'
merged_df_male = pd.merge(accuracy_summary, accuracy_summary_unbiased_classifier_male, on=['True Profession', 'True Gender'], suffixes=('', '_unbiased_classifier'))

merged_df_female = pd.merge(accuracy_summary, accuracy_summary_unbiased_classifier_female, on=['True Profession', 'True Gender'], suffixes=('', '_unbiased_classifier'))

results_male = []
results_female = []


for idx, row in merged_df_female.iterrows():
    
    # Get the current combination's counts
    current_prof = row['True Profession']
    current_gend = row['True Gender']
    current_correct_count = row['Correct Count']
    current_total_count = row['Total Count']
    current_correct_count_unbiased = row['Correct Count_unbiased_classifier'] 
    current_total_count_unbiased = row['Total Count_unbiased_classifier']

    #print(current_prof, current_gend)
    if (current_prof, current_gend) not in minority_indices:
        continue
    
    
    
    # Calculate the metric
    accuracy_minor_group = current_correct_count / current_total_count if current_total_count  > 0 else 0
    accuracy_minor_group_unbiased = current_correct_count_unbiased  / current_total_count_unbiased  if current_total_count_unbiased > 0 else 0
    
    
    # Store the results
    results_female.append({
        'True Profession': current_prof,
        'True Gender': current_gend,
        'acc_minor': accuracy_minor_group,
        'acc_unbiased': accuracy_minor_group_unbiased
    })

for idx, row in merged_df_male.iterrows():
    # Get the current combination's counts
    current_prof = row['True Profession']
    current_gend = row['True Gender']
    current_correct_count = row['Correct Count']
    current_total_count = row['Total Count']
    current_correct_count_unbiased = row['Correct Count_unbiased_classifier'] 
    current_total_count_unbiased = row['Total Count_unbiased_classifier']
    
     # Compute counts for all other combinations
    if (current_prof, current_gend) not in minority_indices:
        continue
    
    
    
    # Calculate the metric
    accuracy_minor_group = current_correct_count / current_total_count if current_total_count  > 0 else 0
    accuracy_minor_group_unbiased = current_correct_count_unbiased  / current_total_count_unbiased  if current_total_count_unbiased > 0 else 0
    
    
    # Store the results
    results_male.append({
        'True Profession': current_prof,
        'True Gender': current_gend,
        'acc_minor': accuracy_minor_group,
        'acc_unbiased': accuracy_minor_group_unbiased
    })
    
    
# Convert results to DataFrame
results_df_male = pd.DataFrame(results_male)
results_df_female = pd.DataFrame(results_female)

mean_minor = results_df_male['acc_minor'].mean()
mean_male = results_df_male['acc_unbiased'].mean()
mean_female = results_df_female['acc_unbiased'].mean()

print(f'Acc_curr_minority: {mean_minor}, Acc_unb_male: {mean_male},  Acc_unb_female: {mean_female}')

print("SP Score (Male):", abs(1-mean_minor/mean_male))
print("SP Score (Female):", abs(1-mean_minor/mean_female))


Length of DataFrame df1: 56
Length of DataFrame df2: 56
Acc_curr_minority: 0.8666404040607689, Acc_unb_male: 0.9970663265306123,  Acc_unb_female: 0.9980867346938774
SP Score (Male): 0.13080967534393917
SP Score (Female): 0.1316983044298493
